# 01 — AnnData, QC, normalization, PCA/UMAP

Goals:
1. Get to know the AnnData structure of the CITE-seq file.
2. Quality control: recompute QC metrics from raw counts and compare with the provided ones.
3. Normalize RNA ourselves (library size + log1p) instead of using the provided size factors
   (see notebook 00).
4. PCA and UMAP for exploration; look at cell-type, site and donor structure.

**Leakage note.** Library-size normalization, log1p and the per-cell CLR of the proteins
are computed per cell and learn nothing from other cells, so they are safe to apply to
all cells. HVG selection and PCA in this notebook are exploratory only. They are **not**
reused as model inputs: from Phase 3 on they are refit on the training split.

In [ ]:
import os, sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from sklearn.utils.sparsefuncs import mean_variance_axis

from citeseq.paths import CITE_H5AD, FIGURES_DIR, PROCESSED_DIR
from citeseq.io import load_cite, split_modalities, csr_nbytes
from citeseq.labels import to_lineage, LINEAGE_ORDER
from citeseq import plotting as P

SEED = 0
P.set_style()
sc.settings.verbosity = 1

# Optional quick test on a random subset: CITESEQ_SUBSET=5000
SUBSET = int(os.environ.get("CITESEQ_SUBSET", 0)) or None
FIG_DIR = FIGURES_DIR if SUBSET is None else PROCESSED_DIR / "test_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("subset:", SUBSET, "| figures ->", FIG_DIR)

## 1. Load raw counts only

The file holds two full matrices (`X` and `layers['counts']`). We load only the raw counts,
streamed from disk in chunks, so memory is roughly halved.

In [ ]:
mask = None
if SUBSET:
    rng = np.random.default_rng(SEED)
    mask = np.zeros(90_261, dtype=bool)
    mask[rng.choice(mask.size, SUBSET, replace=False)] = True

t0 = time.time()
adata = load_cite(CITE_H5AD, matrix="layers/counts", cell_mask=mask)
print(f"loaded in {time.time() - t0:.0f}s, counts matrix uses {csr_nbytes(adata.X) / 1e9:.2f} GB")
adata

## 2. A tour of the AnnData object

`X` is cells x features. `obs` and `var` are tables aligned to its rows and columns.
RNA (GEX) and protein (ADT) share one matrix here; we split them into two objects.
Splitting also makes the 36 feature names that appear as both a gene and a protein
(e.g. `CD14`) unique within each object.

In [ ]:
rna, adt = split_modalities(adata)
del adata
print(rna)
print(adt)
print("\nX is a", type(rna.X).__name__, "with", f"{rna.X.nnz:,}", "stored values;",
      f"{rna.X.nnz / np.prod(rna.shape):.1%} of the entries are non-zero")

In [ ]:
# Slicing keeps X, obs and var aligned: first 5 cells x 6 proteins, as a small dense table
pd.DataFrame(adt.X[:5, :6].toarray(), index=adt.obs_names[:5], columns=adt.var_names[:6])

In [ ]:
rna.obs[["cell_type", "batch", "DonorNumber", "Site", "is_train"]].head()

## 3. Quality control

We recompute the standard metrics from the raw counts:
* `total_counts`: UMIs per cell (library size)
* `n_genes_by_counts`: genes detected
* `pct_counts_mt`: share of UMIs from mitochondrial genes (names starting with `MT-`)

Then we check that they agree with the provided columns.

In [ ]:
rna.var["mt"] = rna.var_names.str.startswith("MT-")
print("mitochondrial genes:", rna.var["mt"].sum())
sc.pp.calculate_qc_metrics(rna, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)

# The provided QC columns were most likely computed before genes were filtered down to
# 13,953, so ours use a smaller gene set. Expected direction: fewer genes detected and a
# slightly higher mitochondrial share (smaller denominator).
o = rna.obs
d_genes = o["n_genes_by_counts"] - o["GEX_n_genes_by_counts"]
d_mt = o["pct_counts_mt"] - o["GEX_pct_counts_mt"]
print("genes detected, ours - provided: median", d_genes.median(), "| ours <= provided in all cells:", bool((d_genes <= 0).all()))
print("% mito, ours - provided: median", round(d_mt.median(), 3), "| ours >= provided in all cells:", bool((d_mt >= -1e-6).all()))
print("correlation, genes detected:", round(np.corrcoef(o["n_genes_by_counts"], o["GEX_n_genes_by_counts"])[0, 1], 4),
      "| % mito:", round(np.corrcoef(o["pct_counts_mt"], o["GEX_pct_counts_mt"])[0, 1], 4))
adt_total = np.asarray(adt.X.sum(axis=1)).ravel()
print("ADT 134-protein total <= ADT_total_counts (which also counts isotypes):",
      bool(np.all(adt_total <= o["ADT_total_counts"].to_numpy() + 1e-6)))

qc = rna.obs[["total_counts", "n_genes_by_counts", "pct_counts_mt",
              "ADT_total_counts", "ADT_iso_count"]]
qc.describe(percentiles=[.01, .5, .99]).T

In [ ]:
batches = sorted(rna.obs["batch"].unique())
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
P.boxplot_by_group(axes[0], rna.obs["total_counts"], rna.obs["batch"], batches, log=True, ylabel="RNA UMIs per cell")
P.boxplot_by_group(axes[1], rna.obs["n_genes_by_counts"], rna.obs["batch"], batches, log=True, ylabel="genes detected")
P.boxplot_by_group(axes[2], rna.obs["pct_counts_mt"], rna.obs["batch"], batches, ylabel="% mitochondrial UMIs")
P.boxplot_by_group(axes[3], rna.obs["ADT_total_counts"], rna.obs["batch"], batches, log=True, ylabel="ADT counts per cell")
fig.suptitle("QC metrics per sample (boxes: quartiles, whiskers: 1.5 IQR, outliers hidden)", fontsize=10)
fig.tight_layout()
fig.savefig(FIG_DIR / "phase1_qc_per_batch.png")

## 4. Normalization of RNA

**Library size.** Divide each cell by its total UMIs and multiply by 10,000 (CP10k), so
cells captured at different depths become comparable.

**log1p.** Count variance grows with the mean, so without a log the most highly
expressed genes dominate every distance and every PCA axis. The plot below shows this:
on the raw scale variance rises steeply with mean; after log1p the trend is much flatter.

In [ ]:
rna.layers["counts"] = rna.X  # same object, no copy
norm = sc.pp.normalize_total(rna, target_sum=1e4, inplace=False)["X"]
# gene mean/variance on the CP10k scale, BEFORE log1p (log1p below works in place)
mean_raw, var_raw = mean_variance_axis(norm, axis=0)
rna.X = norm
sc.pp.log1p(rna)  # in place on X
del norm
print("log-normalized X: max", rna.X.max(), "| memory", f"{csr_nbytes(rna.X) / 1e9:.2f} GB")

# Library-size factor (mean 1) vs the provided scran size factor
lib_sf = rna.obs["total_counts"] / rna.obs["total_counts"].mean()
print("Spearman(library size factor, provided scran size factor):",
      round(lib_sf.corr(rna.obs["GEX_size_factors"], method="spearman"), 3))

In [ ]:
mean_log, var_log = mean_variance_axis(rna.X, axis=0)      # CP10k + log1p
keep = mean_raw > 0

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
ax = axes[0]
ax.scatter(lib_sf, rna.obs["GEX_size_factors"], s=1, c=P.BLUE, alpha=0.3, linewidths=0, rasterized=True)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("library size factor (ours)"); ax.set_ylabel("provided scran size factor")
ax.set_title("Size factors")
ax = axes[1]
ax.scatter(mean_raw[keep], var_raw[keep], s=2, c=P.BLUE, alpha=0.4, linewidths=0, rasterized=True)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("gene mean (CP10k)"); ax.set_ylabel("gene variance")
ax.set_title("Before log: variance grows with mean")
ax = axes[2]
ax.scatter(mean_log[keep], var_log[keep], s=2, c=P.BLUE, alpha=0.4, linewidths=0, rasterized=True)
ax.set_xlabel("gene mean (log1p CP10k)"); ax.set_ylabel("gene variance")
ax.set_title("After log1p")
fig.tight_layout()
fig.savefig(FIG_DIR / "phase1_normalization.png")

## 5. Highly variable genes and PCA (exploratory)

HVGs are selected per sample (`batch_key="batch"`) and combined, so genes that vary only
because of one sample are less likely to be chosen. PCA runs directly on the sparse
log-normalized HVG matrix (centering is done implicitly). We skip unit-variance
scaling, because scaling would turn the matrix into a dense one.

In [ ]:
sc.pp.highly_variable_genes(rna, n_top_genes=2000, batch_key="batch", flavor="seurat")
hvg = rna[:, rna.var["highly_variable"]].copy()
sc.pp.pca(hvg, n_comps=50, svd_solver="arpack", random_state=SEED)
vr = hvg.uns["pca"]["variance_ratio"]
print("HVGs:", hvg.n_vars, "| variance explained by PC1-10:", vr[:10].sum().round(3), "| PC1-50:", vr.sum().round(3))

fig, ax = plt.subplots(figsize=(4.5, 3))
ax.plot(np.arange(1, 51), vr, color=P.BLUE, lw=2, marker="o", ms=3)
ax.set_xlabel("principal component"); ax.set_ylabel("variance ratio")
ax.set_title("PCA scree plot (2,000 HVGs)")
fig.tight_layout()
fig.savefig(FIG_DIR / "phase1_pca_scree.png")

In [ ]:
t0 = time.time()
sc.pp.neighbors(hvg, n_neighbors=15, n_pcs=30, random_state=SEED)
sc.tl.umap(hvg, random_state=SEED)
print(f"neighbors + UMAP: {time.time() - t0:.0f}s")
rna.obsm["X_pca"] = hvg.obsm["X_pca"]
rna.obsm["X_umap"] = hvg.obsm["X_umap"]

## 6. Cell types, sites and donors on the UMAP

Each panel highlights one group in blue over all cells in grey. If a site or donor covers
all the clusters, cells mix across that group in this embedding. If it sits apart, there is
a sample-specific signal (biological or technical). The lineage groups are our own coarse
grouping of the 45 labels, for plotting only.

In [ ]:
rna.obs["lineage"] = pd.Categorical(to_lineage(rna.obs["cell_type"]), categories=LINEAGE_ORDER)
xy = rna.obsm["X_umap"]
fig = P.highlight_grid(xy, rna.obs["lineage"], LINEAGE_ORDER, ncols=4, suptitle="RNA UMAP: lineage")
fig.savefig(FIG_DIR / "phase1_umap_lineage.png")

In [ ]:
fig = P.highlight_grid(xy, rna.obs["Site"], sorted(rna.obs["Site"].unique()), ncols=4, suptitle="RNA UMAP: site")
fig.savefig(FIG_DIR / "phase1_umap_site.png")
donors = sorted(rna.obs["DonorNumber"].unique(), key=lambda d: int(d.replace("donor", "")))
fig = P.highlight_grid(xy, rna.obs["DonorNumber"], donors, ncols=5, suptitle="RNA UMAP: donor")
fig.savefig(FIG_DIR / "phase1_umap_donor.png")

## 7. Cell type x donor

The within-cell-type metrics in later phases need enough cells of a type in the test
donors. How many types reach a minimum count in every donor?

In [ ]:
ct_donor = pd.crosstab(rna.obs["cell_type"], rna.obs["DonorNumber"])[donors]
ct_donor = ct_donor.loc[ct_donor.sum(axis=1).sort_values(ascending=False).index]
for k in (20, 50, 100):
    print(f"cell types with >= {k:>3} cells in every donor: {(ct_donor.min(axis=1) >= k).sum()} / {len(ct_donor)}")
ct_donor

In [ ]:
# Share of each donor's cells per lineage: does the composition differ between donors?
comp = pd.crosstab(rna.obs["DonorNumber"], rna.obs["lineage"], normalize="index")[LINEAGE_ORDER].loc[donors]
(comp * 100).round(1)

In [ ]:
# Save per-cell results for later phases (UMAP is for plotting only)
out = rna.obs[["cell_type", "lineage", "batch", "DonorNumber", "Site",
               "total_counts", "n_genes_by_counts", "pct_counts_mt"]].copy()
out[["umap1", "umap2"]] = rna.obsm["X_umap"]
if SUBSET is None:
    out.to_csv(PROCESSED_DIR / "phase1_cells.csv.gz")
print(out.shape)